# Create the Final Dataset with Weather + Station + POI + Trip Data

### Process
1. Load all CSVs
2. Remove trips with invalid station IDs
3. Extract temporal features from trip timestamps ← Do this here
4. Create complete time skeleton (all hours × all active stations)
5. Aggregate trips by [date, hour, station_id] → rides_started, rides_ended
6. Join to time skeleton (fill nulls with 0) ← Include zero-demand hours
7. Join weather data by [date, hour]
8. Join station metadata by station_id
9. Add cyclical encodings for temporal features
10. Validate data quality
11. Export CSV

### Load CSVs and Initial Inspection

In [1]:
import pandas as pd

bike_trips_df = pd.read_csv('all_bike_trips.csv')
weather_df = pd.read_csv('all_weather.csv')
stations_df = pd.read_csv('final_bike_stations.csv')


In [2]:
print("=== Bike Trips Data ===")
print(f"\nShape: {bike_trips_df.shape}")
print(f"Columns: {bike_trips_df.columns.tolist()}")

print("\n=== Weather Data ===")
print(f"\nShape: {weather_df.shape}")
print(f"Columns: {weather_df.columns.tolist()}")

print("\n=== Stations Data ===")
print(f"\nShape: {stations_df.shape}")
print(f"Columns: {stations_df.columns.tolist()}")

=== Bike Trips Data ===

Shape: (10980291, 10)
Columns: ['Trip Id', 'Start Station Id', 'Start Time', 'Start Station Name', 'End Station Id', 'End Time', 'End Station Name', 'Bike Id', 'User Type', 'Trip Duration']

=== Weather Data ===

Shape: (17544, 16)
Columns: ['Longitude', 'Latitude', 'Station Name', 'Climate ID', 'Temp (°C)', 'Dew Point Temp (°C)', 'Rel Hum (%)', 'Precip. Amount (mm)', 'Wind Dir (10s deg)', 'Wind Spd (km/h)', 'Visibility (km)', 'Stn Press (kPa)', 'Hmdx', 'Wind Chill', 'Weather', 'Date/Time']

=== Stations Data ===

Shape: (1042, 26)
Columns: ['station_id', 'name', 'lat', 'lon', 'capacity', 'address', 'is_charging_station', 'nearby_distance', 'bus', 'subway', 'streetcar', 'train', 'all_public_transport', 'tourism', 'office', 'park', 'healthcare', 'education', 'food_drink', 'commercial', 'cluster_name', 'cluster_east_end', 'cluster_scarborough', 'cluster_uptown', 'cluster_west_end', 'dist_to_union_km']


### Convert Date Columns to Datetime

In [3]:
# Convert datetime columns for trips
bike_trips_df['Start Time'] = pd.to_datetime(bike_trips_df['Start Time'])
bike_trips_df['End Time'] = pd.to_datetime(bike_trips_df['End Time'])

# Convert weather datetime
weather_df['Date/Time'] = pd.to_datetime(weather_df['Date/Time'])

print("Date columns converted to datetime")
print(f"Trips date range: {bike_trips_df['Start Time'].min()} to {bike_trips_df['Start Time'].max()}")
print(f"Weather date range: {weather_df['Date/Time'].min()} to {weather_df['Date/Time'].max()}")


Date columns converted to datetime
Trips date range: 2023-01-01 00:00:00 to 2024-09-30 23:59:00
Weather date range: 2023-01-01 00:00:00 to 2024-12-31 23:00:00


In [4]:
# Check for proper date/time conversion
# Weather Data
spring_dst = weather_df[
    (weather_df['Date/Time'].dt.year == 2023) & 
    (weather_df['Date/Time'].dt.month == 3) & 
    (weather_df['Date/Time'].dt.day == 12) &  # Adjust to actual DST date
    (weather_df['Date/Time'].dt.hour == 2)
]
print(f"Weather at 2 AM during spring DST: {len(spring_dst)}")

# Trips Data
spring_dst = bike_trips_df[
    (bike_trips_df['Start Time'].dt.year == 2023) & 
    (bike_trips_df['Start Time'].dt.month == 3) & 
    (bike_trips_df['Start Time'].dt.day == 12) &  # Adjust to actual DST date
    (bike_trips_df['Start Time'].dt.hour == 2)
]

print(f"Trips at 2 AM during spring DST: {len(spring_dst)}")

Weather at 2 AM during spring DST: 0
Trips at 2 AM during spring DST: 0


### Remove Trips with Invalid Stations

In [5]:
# Set of stations that actually have activity
active_start_stations = set(bike_trips_df['Start Station Id'].unique())
active_end_stations = set(bike_trips_df['End Station Id'].unique())
active_stations = active_start_stations.union(active_end_stations)

print(f"Stations in trips data: {len(active_stations)}")
print(f"Stations in metadata: {len(stations_df)}")

# 2. FILTER STATIONS (The Ghost Fix)
# We overwrite stations_df to only include those that appear in the trips
stations_df = stations_df[stations_df['station_id'].isin(active_stations)]
print(f"Stations after removing ghosts: {len(stations_df)}")

# 3. FILTER TRIPS
valid_ids = set(stations_df['station_id'])
bike_trips_df = bike_trips_df[
    bike_trips_df['Start Station Id'].isin(valid_ids) &
    bike_trips_df['End Station Id'].isin(valid_ids)
]

print(f"Trips after filtering: {len(bike_trips_df):,}")

Stations in trips data: 893
Stations in metadata: 1042
Stations after removing ghosts: 886
Trips after filtering: 10,958,811


### Extract Temporal Features from Trips


In [6]:
# Extract basic temporal features
bike_trips_df['date'] = bike_trips_df['Start Time'].dt.date
bike_trips_df['hour'] = bike_trips_df['Start Time'].dt.hour
bike_trips_df['day_of_week'] = bike_trips_df['Start Time'].dt.dayofweek  # 0=Monday, 6=Sunday
bike_trips_df['day_name'] = bike_trips_df['Start Time'].dt.day_name()
bike_trips_df['month'] = bike_trips_df['Start Time'].dt.month
bike_trips_df['month_name'] = bike_trips_df['Start Time'].dt.month_name()
bike_trips_df['year'] = bike_trips_df['Start Time'].dt.year

# Create derived features
bike_trips_df['is_weekend'] = bike_trips_df['day_of_week'].isin([5, 6])
bike_trips_df['is_am_rush_hour'] = bike_trips_df['hour'].isin([7, 8, 9])
bike_trips_df['is_pm_rush_hour'] = bike_trips_df['hour'].isin([16, 17, 18])

print("Temporal features extracted:")
print(bike_trips_df[['Start Time', 'date', 'hour', 'day_of_week', 'day_name', 
                      'month', 'month_name', 'is_weekend', 'is_am_rush_hour', 'is_pm_rush_hour']].head(10))

Temporal features extracted:
           Start Time        date  hour  day_of_week  day_name  month  \
0 2024-02-01 00:00:00  2024-02-01     0            3  Thursday      2   
1 2024-02-01 00:02:00  2024-02-01     0            3  Thursday      2   
2 2024-02-01 00:02:00  2024-02-01     0            3  Thursday      2   
3 2024-02-01 00:02:00  2024-02-01     0            3  Thursday      2   
4 2024-02-01 00:02:00  2024-02-01     0            3  Thursday      2   
5 2024-02-01 00:03:00  2024-02-01     0            3  Thursday      2   
6 2024-02-01 00:03:00  2024-02-01     0            3  Thursday      2   
7 2024-02-01 00:04:00  2024-02-01     0            3  Thursday      2   
8 2024-02-01 00:05:00  2024-02-01     0            3  Thursday      2   
9 2024-02-01 00:05:00  2024-02-01     0            3  Thursday      2   

  month_name  is_weekend  is_am_rush_hour  is_pm_rush_hour  
0   February       False            False            False  
1   February       False            False    

### Create Complete Time Skeleton

In [7]:
# Get date range from trips
min_date = bike_trips_df['Start Time'].min()
max_date = bike_trips_df['Start Time'].max()

print(f"Creating time skeleton from {min_date} to {max_date}")

# Create all hours in the date range
all_hours = pd.date_range(start=min_date.floor('H'), end=max_date.ceil('H'), freq='H')

# Create all combinations of hours and stations
time_skeleton = pd.DataFrame({
    'datetime': all_hours
})

# Add station dimension
time_skeleton = time_skeleton.merge(
    stations_df[['station_id']], 
    how='cross'
)

# Extract temporal features for skeleton
time_skeleton['date'] = time_skeleton['datetime'].dt.date
time_skeleton['hour'] = time_skeleton['datetime'].dt.hour
time_skeleton['day_of_week'] = time_skeleton['datetime'].dt.dayofweek
time_skeleton['day_name'] = time_skeleton['datetime'].dt.day_name()
time_skeleton['month'] = time_skeleton['datetime'].dt.month
time_skeleton['month_name'] = time_skeleton['datetime'].dt.month_name()
time_skeleton['year'] = time_skeleton['datetime'].dt.year
time_skeleton['is_weekend'] = time_skeleton['day_of_week'].isin([5, 6])
time_skeleton['is_am_rush_hour'] = time_skeleton['hour'].isin([7, 8, 9])
time_skeleton['is_pm_rush_hour'] = time_skeleton['hour'].isin([16, 17, 18])


print(f"\nTime skeleton created:")
print(f"Total rows (hours × stations): {len(time_skeleton):,}")
print(f"Unique hours: {time_skeleton['datetime'].nunique():,}")
print(f"Unique stations: {time_skeleton['station_id'].nunique():,}")
print("\nSample:")
print(time_skeleton.head(10))

Creating time skeleton from 2023-01-01 00:00:00 to 2024-09-30 23:59:00


/var/folders/8k/klw6h73n6vbfxmtrqkgv_fmc0000gn/T/ipykernel_78036/149564608.py:8: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  all_hours = pd.date_range(start=min_date.floor('H'), end=max_date.ceil('H'), freq='H')



Time skeleton created:
Total rows (hours × stations): 13,588,582
Unique hours: 15,337
Unique stations: 886

Sample:
    datetime  station_id        date  hour  day_of_week day_name  month  \
0 2023-01-01        7000  2023-01-01     0            6   Sunday      1   
1 2023-01-01        7001  2023-01-01     0            6   Sunday      1   
2 2023-01-01        7002  2023-01-01     0            6   Sunday      1   
3 2023-01-01        7003  2023-01-01     0            6   Sunday      1   
4 2023-01-01        7005  2023-01-01     0            6   Sunday      1   
5 2023-01-01        7006  2023-01-01     0            6   Sunday      1   
6 2023-01-01        7007  2023-01-01     0            6   Sunday      1   
7 2023-01-01        7008  2023-01-01     0            6   Sunday      1   
8 2023-01-01        7009  2023-01-01     0            6   Sunday      1   
9 2023-01-01        7010  2023-01-01     0            6   Sunday      1   

  month_name  year  is_weekend  is_am_rush_hour  is_pm_ru

### Aggregate Trips Started by Hour and Station


In [8]:
# Count trips started per hour per station
trips_started = bike_trips_df.groupby(['date', 'hour', 'Start Station Id']).size().reset_index(name='trips_started')
trips_started.rename(columns={'Start Station Id': 'station_id'}, inplace=True)

print(f"Trips started aggregated: {len(trips_started):,} rows")
print(trips_started.head(10))
print(f"\nTrips started stats:")
print(trips_started['trips_started'].describe())


Trips started aggregated: 3,969,028 rows
         date  hour  station_id  trips_started
0  2023-01-01     0        7000              2
1  2023-01-01     0        7006              1
2  2023-01-01     0        7014              1
3  2023-01-01     0        7015              2
4  2023-01-01     0        7020              1
5  2023-01-01     0        7021              5
6  2023-01-01     0        7022              2
7  2023-01-01     0        7030              2
8  2023-01-01     0        7033              3
9  2023-01-01     0        7036              1

Trips started stats:
count    3.969028e+06
mean     2.761082e+00
std      2.820485e+00
min      1.000000e+00
25%      1.000000e+00
50%      2.000000e+00
75%      3.000000e+00
max      1.740000e+02
Name: trips_started, dtype: float64


### Aggregate Trips Ended by Hour and Station

In [9]:
# Count trips ended per hour per station
trips_ended = bike_trips_df.groupby(['date', 'hour', 'End Station Id']).size().reset_index(name='trips_ended')
trips_ended.rename(columns={'End Station Id': 'station_id'}, inplace=True)

print(f"Trips ended aggregated: {len(trips_ended):,} rows")
print(trips_ended.head(10))
print(f"\nTrips ended stats:")
print(trips_ended['trips_ended'].describe())

Trips ended aggregated: 3,913,849 rows
         date  hour  station_id  trips_ended
0  2023-01-01     0        7001            1
1  2023-01-01     0        7006            6
2  2023-01-01     0        7014            6
3  2023-01-01     0        7020            1
4  2023-01-01     0        7021            2
5  2023-01-01     0        7022            1
6  2023-01-01     0        7026            1
7  2023-01-01     0        7028            1
8  2023-01-01     0        7030           10
9  2023-01-01     0        7033            2

Trips ended stats:
count    3.913849e+06
mean     2.800009e+00
std      3.149500e+00
min      1.000000e+00
25%      1.000000e+00
50%      2.000000e+00
75%      3.000000e+00
max      2.320000e+02
Name: trips_ended, dtype: float64


### Join Trip Counts to Time Skeleton

In [10]:
# Join trips started
final_df = time_skeleton.merge(
    trips_started,
    on=['date', 'hour', 'station_id'],
    how='left'
)

# Join trips ended
final_df = final_df.merge(
    trips_ended,
    on=['date', 'hour', 'station_id'],
    how='left'
)

# Fill NaN with 0 (hours with no trips)
final_df['trips_started'] = final_df['trips_started'].fillna(0).astype(int)
final_df['trips_ended'] = final_df['trips_ended'].fillna(0).astype(int)

# Calculate net flow
final_df['net_flow'] = final_df['trips_started'] - final_df['trips_ended']

print(f"Final dataframe after joining trip counts: {len(final_df):,} rows")
print(f"\nZero-demand hours:")
print(f"  Hours with 0 trips started: {(final_df['trips_started'] == 0).sum():,} ({(final_df['trips_started'] == 0).sum()/len(final_df)*100:.1f}%)")
print(f"  Hours with 0 trips ended: {(final_df['trips_ended'] == 0).sum():,} ({(final_df['trips_ended'] == 0).sum()/len(final_df)*100:.1f}%)")
print("\nSample with some trips:")
print(final_df[final_df['trips_started'] > 0].head(10))

Final dataframe after joining trip counts: 13,588,582 rows

Zero-demand hours:
  Hours with 0 trips started: 9,619,554 (70.8%)
  Hours with 0 trips ended: 9,674,733 (71.2%)

Sample with some trips:
     datetime  station_id        date  hour  day_of_week day_name  month  \
0  2023-01-01        7000  2023-01-01     0            6   Sunday      1   
5  2023-01-01        7006  2023-01-01     0            6   Sunday      1   
12 2023-01-01        7014  2023-01-01     0            6   Sunday      1   
13 2023-01-01        7015  2023-01-01     0            6   Sunday      1   
18 2023-01-01        7020  2023-01-01     0            6   Sunday      1   
19 2023-01-01        7021  2023-01-01     0            6   Sunday      1   
20 2023-01-01        7022  2023-01-01     0            6   Sunday      1   
28 2023-01-01        7030  2023-01-01     0            6   Sunday      1   
31 2023-01-01        7033  2023-01-01     0            6   Sunday      1   
34 2023-01-01        7036  2023-01-01     

In [11]:
monthly_trips_agg = final_df.groupby(['year', 'month', 'month_name'])['trips_started'].sum().reset_index()
monthly_trips_agg.rename(columns={'trips_started': 'total_trips'}, inplace=True)
print("Trips per month (from aggregated data):")
print(monthly_trips_agg)
print(monthly_trips_agg['total_trips'].sum())

Trips per month (from aggregated data):
    year  month month_name  total_trips
0   2023      1    January       178409
1   2023      2   February       171381
2   2023      3      March       222718
3   2023      4      April       376084
4   2023      5        May       581028
5   2023      6       June       656115
6   2023      7       July       727739
7   2023      8     August       751584
8   2023      9  September       746562
9   2023     10    October       595520
10  2023     11   November       391358
11  2023     12   December       255294
12  2024      1    January       204039
13  2024      2   February       260633
14  2024      3      March       311051
15  2024      4      April       400999
16  2024      5        May       680106
17  2024      6       June       760752
18  2024      7       July       896068
19  2024      8     August       898108
20  2024      9  September       893263
21  2024     10    October            0
10958811


### Prepare Weather Data for Joining

In [12]:
# Extract date and hour from weather data
weather_df['date'] = weather_df['Date/Time'].dt.date
weather_df['hour'] = weather_df['Date/Time'].dt.hour

# Check for duplicate weather records
duplicates = weather_df.groupby(['date', 'hour']).size()
if (duplicates > 1).any():
    print(f"Warning: Found {(duplicates > 1).sum()} duplicate weather records")
    # Keep first occurrence or average them
    weather_df = weather_df.groupby(['date', 'hour']).first().reset_index()

# Select relevant weather columns
weather_columns = ['date', 'hour', 'Temp (°C)', 'Dew Point Temp (°C)', 'Rel Hum (%)', 
                   'Precip. Amount (mm)', 'Wind Spd (km/h)', 'Visibility (km)', 
                   'Stn Press (kPa)']

weather_clean = weather_df[weather_columns].copy()

print(f"Weather data prepared: {len(weather_clean):,} rows")
print(f"Date range: {weather_clean['date'].min()} to {weather_clean['date'].max()}")
print(f"\nMissing values:")
print(weather_clean.isnull().sum())
print("\nSample:")
print(weather_clean.head())

Weather data prepared: 17,542 rows
Date range: 2023-01-01 to 2024-12-31

Missing values:
date                   0
hour                   0
Temp (°C)              0
Dew Point Temp (°C)    0
Rel Hum (%)            0
Precip. Amount (mm)    0
Wind Spd (km/h)        0
Visibility (km)        0
Stn Press (kPa)        0
dtype: int64

Sample:
         date  hour  Temp (°C)  Dew Point Temp (°C)  Rel Hum (%)  \
0  2023-01-01     0        4.0                  1.4         83.0   
1  2023-01-01     1        3.9                  1.9         87.0   
2  2023-01-01     2        3.7                  1.7         87.0   
3  2023-01-01     3        3.5                  1.5         87.0   
4  2023-01-01     4        4.4                  1.1         79.0   

   Precip. Amount (mm)  Wind Spd (km/h)  Visibility (km)  Stn Press (kPa)  
0                  0.0             11.0             16.1           100.24  
1                  0.0             18.0             16.1           100.26  
2                  0.0     

### Join Weather Data

In [13]:
# Join weather to final dataset
final_df = final_df.merge(
    weather_clean,
    on=['date', 'hour'],
    how='left'
)

print(f"After weather join: {len(final_df):,} rows")
print(f"\nMissing weather data:")
print(final_df[['Temp (°C)', 'Rel Hum (%)', 'Precip. Amount (mm)']].isnull().sum())

# Check for any missing weather
missing_weather_count = final_df['Temp (°C)'].isnull().sum()
if missing_weather_count > 0:
    print(f"\nWarning: {missing_weather_count} rows missing weather data")
    print("Consider forward-filling or interpolating these values")
    
print("\nSample:")
print(final_df.head(10))

After weather join: 13,588,582 rows

Missing weather data:
Temp (°C)              1772
Rel Hum (%)            1772
Precip. Amount (mm)    1772
dtype: int64

Consider forward-filling or interpolating these values

Sample:
    datetime  station_id        date  hour  day_of_week day_name  month  \
0 2023-01-01        7000  2023-01-01     0            6   Sunday      1   
1 2023-01-01        7001  2023-01-01     0            6   Sunday      1   
2 2023-01-01        7002  2023-01-01     0            6   Sunday      1   
3 2023-01-01        7003  2023-01-01     0            6   Sunday      1   
4 2023-01-01        7005  2023-01-01     0            6   Sunday      1   
5 2023-01-01        7006  2023-01-01     0            6   Sunday      1   
6 2023-01-01        7007  2023-01-01     0            6   Sunday      1   
7 2023-01-01        7008  2023-01-01     0            6   Sunday      1   
8 2023-01-01        7009  2023-01-01     0            6   Sunday      1   
9 2023-01-01        7010  202

In [14]:
print(final_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13588582 entries, 0 to 13588581
Data columns (total 22 columns):
 #   Column               Dtype         
---  ------               -----         
 0   datetime             datetime64[ns]
 1   station_id           int64         
 2   date                 object        
 3   hour                 int32         
 4   day_of_week          int32         
 5   day_name             object        
 6   month                int32         
 7   month_name           object        
 8   year                 int32         
 9   is_weekend           bool          
 10  is_am_rush_hour      bool          
 11  is_pm_rush_hour      bool          
 12  trips_started        int64         
 13  trips_ended          int64         
 14  net_flow             int64         
 15  Temp (°C)            float64       
 16  Dew Point Temp (°C)  float64       
 17  Rel Hum (%)          float64       
 18  Precip. Amount (mm)  float64       
 19  Wind Spd (km/h)    

### Join Station Metadata

In [15]:
# Join station information
final_df = final_df.merge(
    stations_df,
    on='station_id',
    how='left'
)

print(f"After station join: {len(final_df):,} rows")
print(f"\nColumns in final dataset: {final_df.columns.tolist()}")
print(f"\nMissing station data:")
print(final_df[stations_df.columns].isnull().sum())

print("\nSample:")
print(final_df.info())


After station join: 13,588,582 rows

Columns in final dataset: ['datetime', 'station_id', 'date', 'hour', 'day_of_week', 'day_name', 'month', 'month_name', 'year', 'is_weekend', 'is_am_rush_hour', 'is_pm_rush_hour', 'trips_started', 'trips_ended', 'net_flow', 'Temp (°C)', 'Dew Point Temp (°C)', 'Rel Hum (%)', 'Precip. Amount (mm)', 'Wind Spd (km/h)', 'Visibility (km)', 'Stn Press (kPa)', 'name', 'lat', 'lon', 'capacity', 'address', 'is_charging_station', 'nearby_distance', 'bus', 'subway', 'streetcar', 'train', 'all_public_transport', 'tourism', 'office', 'park', 'healthcare', 'education', 'food_drink', 'commercial', 'cluster_name', 'cluster_east_end', 'cluster_scarborough', 'cluster_uptown', 'cluster_west_end', 'dist_to_union_km']

Missing station data:
station_id                   0
name                         0
lat                          0
lon                          0
capacity                     0
address                 904883
is_charging_station          0
nearby_distance   

In [16]:
final_df_clean = final_df.drop(['address', 'nearby_distance'], axis=1)
print(final_df_clean.info())
print(final_df_clean.columns.tolist())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13588582 entries, 0 to 13588581
Data columns (total 45 columns):
 #   Column                Dtype         
---  ------                -----         
 0   datetime              datetime64[ns]
 1   station_id            int64         
 2   date                  object        
 3   hour                  int32         
 4   day_of_week           int32         
 5   day_name              object        
 6   month                 int32         
 7   month_name            object        
 8   year                  int32         
 9   is_weekend            bool          
 10  is_am_rush_hour       bool          
 11  is_pm_rush_hour       bool          
 12  trips_started         int64         
 13  trips_ended           int64         
 14  net_flow              int64         
 15  Temp (°C)             float64       
 16  Dew Point Temp (°C)   float64       
 17  Rel Hum (%)           float64       
 18  Precip. Amount (mm)   float64       
 19

### Add Cyclical Encodings


In [17]:
import numpy as np

# Add cyclical encodings for temporal features
# Hour of day (0-23)
final_df_clean['hour_sin'] = np.sin(2 * np.pi * final_df_clean['hour'] / 24)
final_df_clean['hour_cos'] = np.cos(2 * np.pi * final_df_clean['hour'] / 24)

# Day of week (0-6)
final_df_clean['dow_sin'] = np.sin(2 * np.pi * final_df_clean['day_of_week'] / 7)
final_df_clean['dow_cos'] = np.cos(2 * np.pi * final_df_clean['day_of_week'] / 7)

# Month (1-12)
final_df_clean['month_sin'] = np.sin(2 * np.pi * final_df_clean['month'] / 12)
final_df_clean['month_cos'] = np.cos(2 * np.pi * final_df_clean['month'] / 12)

print("Cyclical encodings added")
print(final_df_clean[['hour', 'hour_sin', 'hour_cos', 'day_of_week', 'dow_sin', 'dow_cos']].head())
print(final_df_clean.columns.tolist())

Cyclical encodings added
   hour  hour_sin  hour_cos  day_of_week   dow_sin  dow_cos
0     0       0.0       1.0            6 -0.781831  0.62349
1     0       0.0       1.0            6 -0.781831  0.62349
2     0       0.0       1.0            6 -0.781831  0.62349
3     0       0.0       1.0            6 -0.781831  0.62349
4     0       0.0       1.0            6 -0.781831  0.62349
['datetime', 'station_id', 'date', 'hour', 'day_of_week', 'day_name', 'month', 'month_name', 'year', 'is_weekend', 'is_am_rush_hour', 'is_pm_rush_hour', 'trips_started', 'trips_ended', 'net_flow', 'Temp (°C)', 'Dew Point Temp (°C)', 'Rel Hum (%)', 'Precip. Amount (mm)', 'Wind Spd (km/h)', 'Visibility (km)', 'Stn Press (kPa)', 'name', 'lat', 'lon', 'capacity', 'is_charging_station', 'bus', 'subway', 'streetcar', 'train', 'all_public_transport', 'tourism', 'office', 'park', 'healthcare', 'education', 'food_drink', 'commercial', 'cluster_name', 'cluster_east_end', 'cluster_scarborough', 'cluster_uptown', 'clust

### Data Quality Validation

In [18]:
print("=== DATA QUALITY VALIDATION ===\n")

# Check for nulls
print("Missing values by column:")
null_counts = final_df_clean.isnull().sum()
print(null_counts[null_counts > 0])

# Check date range
print(f"\nDate range: {final_df_clean['datetime'].min()} to {final_df_clean['datetime'].max()}")

# Check target variables
print(f"\nTarget variable statistics:")
print(f"Trips started - Mean: {final_df_clean['trips_started'].mean():.2f}, Max: {final_df_clean['trips_started'].max()}")
print(f"Trips ended - Mean: {final_df_clean['trips_ended'].mean():.2f}, Max: {final_df_clean['trips_ended'].max()}")
print(f"Net flow - Mean: {final_df_clean['net_flow'].mean():.2f}, Min: {final_df_clean['net_flow'].min()}, Max: {final_df_clean['net_flow'].max()}")

# Check for duplicates
duplicates = final_df_clean.duplicated(subset=['datetime', 'station_id']).sum()
print(f"\nDuplicate rows (datetime + station_id): {duplicates}")

# Summary stats
print(f"\nFinal dataset shape: {final_df_clean.shape}")
print(f"Total hours × stations: {len(final_df_clean):,}")
print(f"Unique stations: {final_df_clean['station_id'].nunique()}")
print(f"Unique hours: {final_df_clean['datetime'].nunique()}")


=== DATA QUALITY VALIDATION ===

Missing values by column:
Temp (°C)              1772
Dew Point Temp (°C)    1772
Rel Hum (%)            1772
Precip. Amount (mm)    1772
Wind Spd (km/h)        1772
Visibility (km)        1772
Stn Press (kPa)        1772
dtype: int64

Date range: 2023-01-01 00:00:00 to 2024-10-01 00:00:00

Target variable statistics:
Trips started - Mean: 0.81, Max: 174
Trips ended - Mean: 0.81, Max: 232
Net flow - Mean: 0.00, Min: -228, Max: 171

Duplicate rows (datetime + station_id): 0

Final dataset shape: (13588582, 51)
Total hours × stations: 13,588,582
Unique stations: 886
Unique hours: 15337


### Quick fix based on data validation
1. Investigate missing weather data
2. Remove missing rows (caused by DST not being accounted for)
3. Drop October data

In [19]:
# Find rows with missing weather data
missing_weather = final_df_clean[final_df_clean['Temp (°C)'].isnull()]

print(f"\n=== INVESTIGATING MISSING WEATHER DATA ===")
print(f"Total rows with missing weather: {len(missing_weather):,}")
print(f"Percentage of dataset: {len(missing_weather)/len(final_df_clean)*100:.2f}%")

# Check date range of missing data
print(f"\nDate range of missing weather:")
print(f"  Earliest: {missing_weather['datetime'].min()}")
print(f"  Latest: {missing_weather['datetime'].max()}")

# Check if missing data is clustered on specific dates
print(f"\nMissing weather by date:")
missing_by_date = missing_weather.groupby('date').size().reset_index(name='missing_count')
print(missing_by_date.sort_values('missing_count', ascending=False).head(20))

# Check if missing data is clustered on specific hours
print(f"\nMissing weather by hour of day:")
missing_by_hour = missing_weather.groupby('hour').size().reset_index(name='missing_count')
print(missing_by_hour.sort_values('hour'))

# Show sample of missing rows
print(f"\nSample of rows with missing weather data:")
print(missing_weather[['datetime', 'station_id', 'date', 'hour', 'trips_started', 
                        'trips_ended', 'Temp (°C)']].head(20))

# Check if these dates exist in the original weather data
print(f"\nChecking original weather data...")
missing_dates = missing_weather['date'].unique()
print(f"Unique dates with missing weather: {len(missing_dates)}")
print(f"Sample missing dates: {missing_dates[:10]}")

# Check if these dates/hours exist in weather_df
for date in missing_dates[:5]:
    weather_on_date = weather_df[weather_df['date'] == date]
    print(f"\nWeather records for {date}: {len(weather_on_date)} hours")
    if len(weather_on_date) > 0:
        print(f"  Hours available: {sorted(weather_on_date['hour'].unique())}")


=== INVESTIGATING MISSING WEATHER DATA ===
Total rows with missing weather: 1,772
Percentage of dataset: 0.01%

Date range of missing weather:
  Earliest: 2023-03-12 02:00:00
  Latest: 2024-03-10 02:00:00

Missing weather by date:
         date  missing_count
0  2023-03-12            886
1  2024-03-10            886

Missing weather by hour of day:
   hour  missing_count
0     2           1772

Sample of rows with missing weather data:
                   datetime  station_id        date  hour  trips_started  \
1490252 2023-03-12 02:00:00        7000  2023-03-12     2              0   
1490253 2023-03-12 02:00:00        7001  2023-03-12     2              0   
1490254 2023-03-12 02:00:00        7002  2023-03-12     2              0   
1490255 2023-03-12 02:00:00        7003  2023-03-12     2              0   
1490256 2023-03-12 02:00:00        7005  2023-03-12     2              0   
1490257 2023-03-12 02:00:00        7006  2023-03-12     2              0   
1490258 2023-03-12 02:00:00

In [20]:
# Check counts before dropping
before_count = len(final_df_clean)
dst_rows = final_df_clean[final_df_clean['Temp (°C)'].isnull()]
dst_count = len(dst_rows)

print(f"Rows with missing weather (DST spring forward): {dst_count:,}")
print(f"Dates affected: {dst_rows['date'].unique()}")

# Drop rows with missing weather data (the non-existent 2 AM hours)
final_df_clean = final_df_clean[final_df_clean['Temp (°C)'].notna()]

after_count = len(final_df_clean)
print(f"\nRows before: {before_count:,}")
print(f"Rows after: {after_count:,}")
print(f"Rows dropped: {before_count - after_count:,}")

# Verify no more missing weather
print(f"\nRemaining missing weather values:")
print(final_df_clean[['Temp (°C)', 'Dew Point Temp (°C)', 'Rel Hum (%)']].isnull().sum())

Rows with missing weather (DST spring forward): 1,772
Dates affected: [datetime.date(2023, 3, 12) datetime.date(2024, 3, 10)]

Rows before: 13,588,582
Rows after: 13,586,810
Rows dropped: 1,772

Remaining missing weather values:
Temp (°C)              0
Dew Point Temp (°C)    0
Rel Hum (%)            0
dtype: int64


In [21]:
oct_1_count = len(final_df_clean[final_df_clean['date'] == pd.to_datetime('2024-10-01').date()])
print(f"Rows on 2024-10-01: {oct_1_count:,}")
print(final_df_clean[final_df_clean['date'] == pd.to_datetime('2024-10-01').date()]['trips_started'].sum())

final_df_cleaned = final_df_clean[final_df_clean['date'] != pd.to_datetime('2024-10-01').date()]

Rows on 2024-10-01: 886
0


### Export to CSV


In [22]:
print("=== FINAL DATA QUALITY VALIDATION ===\n")

# Check for nulls
print("Missing values by column:")
null_counts = final_df_clean.isnull().sum()
print(null_counts[null_counts > 0])

# Check date range
print(f"\nDate range: {final_df_clean['datetime'].min()} to {final_df_clean['datetime'].max()}")

# Check target variables
print(f"\nTarget variable statistics:")
print(f"Trips started - Mean: {final_df_clean['trips_started'].mean():.2f}, Max: {final_df_clean['trips_started'].max()}")
print(f"Trips ended - Mean: {final_df_clean['trips_ended'].mean():.2f}, Max: {final_df_clean['trips_ended'].max()}")
print(f"Net flow - Mean: {final_df_clean['net_flow'].mean():.2f}, Min: {final_df_clean['net_flow'].min()}, Max: {final_df_clean['net_flow'].max()}")

# Check for duplicates
duplicates = final_df_clean.duplicated(subset=['datetime', 'station_id']).sum()
print(f"\nDuplicate rows (datetime + station_id): {duplicates}")

# Summary stats
print(f"\nFinal dataset shape: {final_df_clean.shape}")
print(f"Total hours × stations: {len(final_df_clean):,}")
print(f"Unique stations: {final_df_clean['station_id'].nunique()}")
print(f"Unique hours: {final_df_clean['datetime'].nunique()}")


=== FINAL DATA QUALITY VALIDATION ===

Missing values by column:
Series([], dtype: int64)

Date range: 2023-01-01 00:00:00 to 2024-10-01 00:00:00

Target variable statistics:
Trips started - Mean: 0.81, Max: 174
Trips ended - Mean: 0.81, Max: 232
Net flow - Mean: 0.00, Min: -228, Max: 171

Duplicate rows (datetime + station_id): 0

Final dataset shape: (13586810, 51)
Total hours × stations: 13,586,810
Unique stations: 886
Unique hours: 15335


In [23]:
print("=== DATAFRAME INFO ===")
print(final_df_cleaned.info())

=== DATAFRAME INFO ===
<class 'pandas.core.frame.DataFrame'>
Index: 13585924 entries, 0 to 13587695
Data columns (total 51 columns):
 #   Column                Dtype         
---  ------                -----         
 0   datetime              datetime64[ns]
 1   station_id            int64         
 2   date                  object        
 3   hour                  int32         
 4   day_of_week           int32         
 5   day_name              object        
 6   month                 int32         
 7   month_name            object        
 8   year                  int32         
 9   is_weekend            bool          
 10  is_am_rush_hour       bool          
 11  is_pm_rush_hour       bool          
 12  trips_started         int64         
 13  trips_ended           int64         
 14  net_flow              int64         
 15  Temp (°C)             float64       
 16  Dew Point Temp (°C)   float64       
 17  Rel Hum (%)           float64       
 18  Precip. Amount (mm)   

### Data Size Optimization and Renaming

In [24]:
print("=== CREATING OPTIMIZED COPY ===\n")

# Create a copy to work with
final_df_optimized = final_df_cleaned.copy()

# Check memory before
memory_before = final_df_optimized.memory_usage(deep=True).sum() / 1024**2
print(f"Memory before optimization: {memory_before:.2f} MB")

# 1. OPTIMIZE DATA TYPES
# Integers - use smallest type that fits the range
final_df_optimized['station_id'] = final_df_optimized['station_id'].astype('int16')  # Max ~32k stations
final_df_optimized['hour'] = final_df_optimized['hour'].astype('int8')  # 0-23
final_df_optimized['day_of_week'] = final_df_optimized['day_of_week'].astype('int8')  # 0-6
final_df_optimized['month'] = final_df_optimized['month'].astype('int8')  # 1-12
final_df_optimized['year'] = final_df_optimized['year'].astype('int16')  # 2023-2024

# Trip counts - check max values first
print(f"\nTrip count ranges:")
print(f"  trips_started max: {final_df_optimized['trips_started'].max()}")
print(f"  trips_ended max: {final_df_optimized['trips_ended'].max()}")
print(f"  net_flow range: [{final_df_optimized['net_flow'].min()}, {final_df_optimized['net_flow'].max()}]")

# Use int16 if max < 32767, else int32
if final_df_optimized['trips_started'].max() < 32767:
    final_df_optimized['trips_started'] = final_df_optimized['trips_started'].astype('int16')
else:
    final_df_optimized['trips_started'] = final_df_optimized['trips_started'].astype('int32')
    
if final_df_optimized['trips_ended'].max() < 32767:
    final_df_optimized['trips_ended'] = final_df_optimized['trips_ended'].astype('int16')
else:
    final_df_optimized['trips_ended'] = final_df_optimized['trips_ended'].astype('int32')

final_df_optimized['net_flow'] = final_df_optimized['net_flow'].astype('int16')

# Capacity
final_df_optimized['capacity'] = final_df_optimized['capacity'].astype('int16')

# POI counts - use int16 (unlikely to have >32k POIs near a station)
poi_cols = ['bus', 'subway', 'streetcar', 'train', 'all_public_transport', 
            'tourism', 'office', 'park', 'healthcare', 'education', 
            'food_drink', 'commercial']
for col in poi_cols:
    final_df_optimized[col] = final_df_optimized[col].astype('int16')

# Float columns - use float32 instead of float64
float_cols = ['Temp (°C)', 'Dew Point Temp (°C)', 'Rel Hum (%)', 'Precip. Amount (mm)',
              'Wind Spd (km/h)', 'Visibility (km)', 'Stn Press (kPa)',
              'lat', 'lon', 'dist_to_union_km',
              'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos']
for col in float_cols:
    final_df_optimized[col] = final_df_optimized[col].astype('float32')

# Cluster indicators
cluster_cols = ['cluster_east_end', 'cluster_scarborough', 'cluster_uptown', 'cluster_west_end']
for col in cluster_cols:
    final_df_optimized[col] = final_df_optimized[col].astype('bool')

# Explicitly cast booleans
bool_cols = ['is_weekend', 'is_am_rush_hour', 'is_pm_rush_hour', 'is_charging_station']
for col in bool_cols:
     final_df_optimized[col] = final_df_optimized[col].astype('bool')

# 2. RENAME COLUMNS TO SNAKE_CASE AND REMOVE SPECIAL CHARACTERS
rename_map = {
    # Weather columns - remove special characters
    'Temp (°C)': 'temp_c',
    'Dew Point Temp (°C)': 'dew_point_c',
    'Rel Hum (%)': 'rel_humidity_pct',
    'Precip. Amount (mm)': 'precip_mm',
    'Wind Spd (km/h)': 'wind_speed_kmh',
    'Visibility (km)': 'visibility_km',
    'Stn Press (kPa)': 'pressure_kpa',
    
    # Station columns - more descriptive
    'name': 'station_name',
    'lat': 'latitude',
    'lon': 'longitude',
    
    # POI columns - clearer naming
    'all_public_transport': 'public_transport_total',
    'food_drink': 'restaurants_bars',
}

final_df_optimized.rename(columns=rename_map, inplace=True)

# 3. DROP REDUNDANT COLUMNS
columns_to_drop = [
    'day_name',      # Redundant with day_of_week
    'month_name',    # Redundant with month
    'cluster_name',  # Redundant with cluster_* indicators
]

print("\n=== DROPPING REDUNDANT COLUMNS ===")
print(f"Columns to drop: {columns_to_drop}")

final_df_optimized = final_df_optimized.drop(columns=columns_to_drop)

# Check memory after
memory_after = final_df_optimized.memory_usage(deep=True).sum() / 1024**2
memory_saved = memory_before - memory_after
savings_pct = (memory_saved / memory_before) * 100

print(f"\n=== OPTIMIZATION RESULTS ===")
print(f"Memory before: {memory_before:.2f} MB")
print(f"Memory after: {memory_after:.2f} MB")
print(f"Memory saved: {memory_saved:.2f} MB ({savings_pct:.1f}% reduction)")
print(f"\nOriginal shape: {final_df_cleaned.shape}")
print(f"Optimized shape: {final_df_optimized.shape}")

# Show new dtypes
print("\n=== OPTIMIZED DATA TYPES ===")
print(final_df_optimized.info())

print("\n✓ final_df_cleaned remains unchanged")
print("✓ final_df_optimized is ready for modeling")


=== CREATING OPTIMIZED COPY ===

Memory before optimization: 8388.84 MB

Trip count ranges:
  trips_started max: 174
  trips_ended max: 232
  net_flow range: [-228, 171]

=== DROPPING REDUNDANT COLUMNS ===
Columns to drop: ['day_name', 'month_name', 'cluster_name']

=== OPTIMIZATION RESULTS ===
Memory before: 8388.84 MB
Memory after: 3244.24 MB
Memory saved: 5144.60 MB (61.3% reduction)

Original shape: (13585924, 51)
Optimized shape: (13585924, 48)

=== OPTIMIZED DATA TYPES ===
<class 'pandas.core.frame.DataFrame'>
Index: 13585924 entries, 0 to 13587695
Data columns (total 48 columns):
 #   Column                  Dtype         
---  ------                  -----         
 0   datetime                datetime64[ns]
 1   station_id              int16         
 2   date                    object        
 3   hour                    int8          
 4   day_of_week             int8          
 5   month                   int8          
 6   year                    int16         
 7   is_we

### Apply Log Transformations
Right-skewed features and the target variable are log-transformed (log1p) to improve model performance.

In [25]:
import numpy as np

# 1. Define skewed features
skewed_features = ['precip_mm', 'dist_to_union_km',
                   'tourism', 'office', 'park', 
                   'healthcare', 'education', 'restaurants_bars', 'commercial']

# 2. Apply log1p to features
# Only apply to columns that exist in the dataframe
cols_to_transform = [col for col in skewed_features if col in final_df_optimized.columns]
print(f"Applying log1p transformation to {len(cols_to_transform)} features...")
final_df_optimized[cols_to_transform] = np.log1p(final_df_optimized[cols_to_transform])

# 3. Apply log1p to target
print("Applying log1p transformation to target: trips_started")
final_df_optimized['trips_started'] = np.log1p(final_df_optimized['trips_started'])

# 4. Verify
print(f"Shape: {final_df_optimized.shape}")
print("Sample of transformed target:")
print(final_df_optimized['trips_started'].head())

Applying log1p transformation to 9 features...
Applying log1p transformation to target: trips_started
Shape: (13585924, 48)
Sample of transformed target:
0    1.098612
1    0.000000
2    0.000000
3    0.000000
4    0.000000
Name: trips_started, dtype: float32


In [26]:
# Export final dataset
output_filename = 'final_bike_demand_dataset.csv'
final_df_optimized.to_csv(output_filename, index=False)

print(f"Dataset exported to: {output_filename}")
print(f"File size: {final_df_optimized.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print(f"\nColumns ({len(final_df_optimized.columns)}):")
for i, col in enumerate(final_df_optimized.columns, 1):
    print(f"{i}. {col}")


Dataset exported to: final_bike_demand_dataset.csv
File size: 3451.55 MB

Columns (48):
1. datetime
2. station_id
3. date
4. hour
5. day_of_week
6. month
7. year
8. is_weekend
9. is_am_rush_hour
10. is_pm_rush_hour
11. trips_started
12. trips_ended
13. net_flow
14. temp_c
15. dew_point_c
16. rel_humidity_pct
17. precip_mm
18. wind_speed_kmh
19. visibility_km
20. pressure_kpa
21. station_name
22. latitude
23. longitude
24. capacity
25. is_charging_station
26. bus
27. subway
28. streetcar
29. train
30. public_transport_total
31. tourism
32. office
33. park
34. healthcare
35. education
36. restaurants_bars
37. commercial
38. cluster_east_end
39. cluster_scarborough
40. cluster_uptown
41. cluster_west_end
42. dist_to_union_km
43. hour_sin
44. hour_cos
45. dow_sin
46. dow_cos
47. month_sin
48. month_cos
